In [21]:
import pandas as pd
import matplotlib.pyplot as plt

In [22]:
import os
import pandas as pd

# Path ke folder yang berisi file-file DataFrame
folder_path = '../comodity-price-prediction-penyisihan-arkavidia-9/Harga Bahan Pangan/train'

# Dictionary untuk menyimpan semua DataFrame dengan nama file sebagai key
df_dict = {}

# Loop melalui semua file dalam folder
for file_name in os.listdir(folder_path):
    # Pastikan file yang dibaca adalah file yang diinginkan (misalnya, CSV)
    if file_name.endswith('.csv'):
        # Buat path lengkap ke file
        file_path = os.path.join(folder_path, file_name)
        
        # Baca file dan simpan ke dictionary dengan nama file sebagai key
        df = pd.read_csv(file_path)
        df_dict[file_name] = df  # Gunakan nama file sebagai key


In [23]:
df_bawang_merah = df_dict['Bawang Merah.csv']
df_bawang_putih = df_dict['Bawang Putih Bonggol.csv']
df_beras_medium = df_dict['Beras Medium.csv']
df_beras_premium = df_dict['Beras Premium.csv']
df_cabai_merah_keriting = df_dict['Cabai Merah Keriting.csv']
df_cabai_rawit_merah = df_dict['Cabai Rawit Merah.csv']
df_daging_ayam = df_dict['Daging Ayam Ras.csv']
df_daging_sapi = df_dict['Daging Sapi Murni.csv']
df_gula = df_dict['Gula Konsumsi.csv']
df_minyak_goreng_curah = df_dict['Minyak Goreng Curah.csv']
df_minyak_goreng_kemasan = df_dict['Minyak Goreng Kemasan Sederhana.csv']
df_telur_ayam = df_dict['Telur Ayam Ras.csv']
df_tepung_terigu = df_dict['Tepung Terigu (Curah).csv']

In [24]:
import pandas as pd

# List semua dataframe
df_list = [
    df_bawang_merah, df_bawang_putih, df_beras_medium, df_beras_premium, df_cabai_merah_keriting,
    df_cabai_rawit_merah, df_daging_ayam, df_daging_sapi, df_gula, df_minyak_goreng_curah,
    df_minyak_goreng_kemasan, df_telur_ayam, df_tepung_terigu
]

# Loop setiap dataframe dan lakukan interpolasi
for i in range(len(df_list)):
    df = df_list[i]  # Ambil dataframe

    # Pisahkan kolom tanggal (asumsi ada kolom 'date' atau 'tanggal')
    date_cols = df.select_dtypes(include=['datetime64', 'object'])  # Deteksi kolom tanggal
    num_cols = df.select_dtypes(include=['number'])  # Ambil hanya kolom numerik

    # Interpolasi hanya untuk kolom numerik
    df_num_imputed = num_cols.interpolate(method='linear', limit_direction='both')

    # Gabungkan kembali kolom tanggal dengan hasil interpolasi
    df_list[i] = pd.concat([date_cols.reset_index(drop=True), df_num_imputed.reset_index(drop=True)], axis=1)


In [25]:
df_dict = {
    'Bawang Merah': df_list[0],
    'Bawang Putih Bonggol': df_list[1],
    'Beras Medium': df_list[2],
    'Beras Premium': df_list[3],
    'Cabai Merah Keriting': df_list[4],
    'Cabai Rawit Merah': df_list[5],
    'Daging Ayam Ras': df_list[6],
    'Daging Sapi Murni': df_list[7],
    'Gula Konsumsi': df_list[8],
    'Minyak Goreng Curah': df_list[9],
    'Minyak Goreng Kemasan Sederhana': df_list[10],
    'Telur Ayam Ras': df_list[11],
    'Tepung Terigu (Curah)': df_list[12]
}


In [26]:
df_PCA_global = pd.read_csv('PCA_Dataset/PCA_Exchange.csv')
df_PCA_exchange = pd.read_csv('PCA_Dataset/PCA_global_commo.csv')

In [27]:
df_PCA_exchange.head(5)

,Crude Oil WTI Futures,Natural Gas Futures,Newcastle Coal Futures,Palm Oil Futures,US Sugar 11 Futures,US Wheat Futures
0,-1.310918,-0.084197,-1.321539,1.831232,-1.668793,0.514505
1,-1.310918,-0.084197,-1.321539,1.831232,-1.668793,0.514505
2,-1.310918,-0.084197,-1.321539,1.831232,-1.668793,0.514505
3,-1.162784,-0.087038,-0.963294,1.803851,-1.699332,0.514505
4,-1.037087,-0.009913,-0.854299,2.057301,-1.821628,0.503778


In [28]:
df_PCA_global.head(5)

,MYRUSD,SGDUSD,THBUSD,USDIDR
0,4.419041,0.376185,3.378026,-4.079549
1,4.419041,0.376185,3.378026,-4.079549
2,4.419041,0.376185,3.378026,-4.079549
3,4.367445,-0.014675,3.198768,-3.815928
4,4.204347,-0.128121,3.279147,-3.543750


In [29]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Dictionary untuk menyimpan model dan prediksi
models = {}
predictions = []
mape_scores = []
predict_start = "2024-10-01"
predict_end = "2024-12-31"

# Buat daftar tanggal untuk prediksi
future_dates = pd.date_range(start=predict_start, end=predict_end)

# Loop setiap komoditas dalam df_dict
for komoditas, df_komoditas in df_dict.items():
    print(f"\n🚀 Mengolah komoditas: {komoditas}")

    # Drop kolom "Date" jika ada
    if "Date" in df_komoditas.columns:
        df_komoditas = df_komoditas.drop(columns=["Date"])

    # Loop setiap provinsi dalam df_komoditas
    for provinsi in df_komoditas.columns:
        print(f"  → Menggabungkan data untuk provinsi: {provinsi}...")

        # Ambil hanya data provinsi tersebut
        df_prov = df_komoditas[[provinsi]].rename(columns={provinsi: 'y'})

        # Gabungkan dengan PCA Global dan PCA Exchange
        df_merged = pd.concat([df_prov, df_PCA_global, df_PCA_exchange], axis=1)

        # Pisahkan fitur (X) dan target harga (y)
        X = df_merged.iloc[:, 1:]  # Semua kolom kecuali harga komoditas
        y = df_merged.iloc[:, 0]   # Kolom pertama sebagai target

        # Pisahkan training & testing
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

        # Inisialisasi dan latih model XGBoost
        model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
        model.fit(X_train, y_train)

        # Simpan model
        models[(komoditas, provinsi)] = model

        # Evaluasi model
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        mape = mean_absolute_percentage_error(y_test, y_pred)
        mape_scores.append(mape)  # Simpan MAPE
        print(f"    → MAE untuk {komoditas} - {provinsi}: {mae:.4f}")
        print(f"    → MAPE untuk {komoditas} - {provinsi}: {mape:.4f}")

        # Mulai prediksi ke masa depan dengan iterasi (mirip LSTM)
        last_known_data = X.iloc[-1, :].values.reshape(1, -1)  # Ambil data terakhir
        future_preds = []

        for _ in range(len(future_dates)):
            next_pred = model.predict(last_known_data)[0]  # Prediksi harga hari ini
            future_preds.append(next_pred)

            # Update last_known_data → gantikan harga sebelumnya dengan prediksi baru
            last_known_data = np.roll(last_known_data, -1)  # Geser ke kiri
            last_known_data[0, -1] = next_pred  # Masukkan prediksi ke fitur terakhir

        # Simpan hasil prediksi ke dalam list
        for date, pred in zip(future_dates, future_preds):
            predictions.append({
                'id': f"{komoditas}/{provinsi}/{date.strftime('%Y-%m-%d')}",
                'price': pred  # Harga dibulatkan 4 desimal
            })

# Simpan hasil prediksi ke CSV
submission_df = pd.DataFrame(predictions)
submission_df.to_csv('submission_xgboost_iterative.csv', index=False)
print("\n✅ Hasil prediksi disimpan dalam 'submission_xgboost_iterative.csv'")



🚀 Mengolah komoditas: Bawang Merah
  → Menggabungkan data untuk provinsi: Aceh...
    → MAE untuk Bawang Merah - Aceh: 11408.2818
    → MAPE untuk Bawang Merah - Aceh: 0.2678
  → Menggabungkan data untuk provinsi: Bali...
    → MAE untuk Bawang Merah - Bali: 7599.4743
    → MAPE untuk Bawang Merah - Bali: 0.2176
  → Menggabungkan data untuk provinsi: Banten...
    → MAE untuk Bawang Merah - Banten: 12226.0398
    → MAPE untuk Bawang Merah - Banten: 0.3181
  → Menggabungkan data untuk provinsi: Bengkulu...
    → MAE untuk Bawang Merah - Bengkulu: 10966.3447
    → MAPE untuk Bawang Merah - Bengkulu: 0.2785
  → Menggabungkan data untuk provinsi: DI Yogyakarta...
    → MAE untuk Bawang Merah - DI Yogyakarta: 12616.3313
    → MAPE untuk Bawang Merah - DI Yogyakarta: 0.3893
  → Menggabungkan data untuk provinsi: DKI Jakarta...
    → MAE untuk Bawang Merah - DKI Jakarta: 14897.9394
    → MAPE untuk Bawang Merah - DKI Jakarta: 0.3164
  → Menggabungkan data untuk provinsi: Gorontalo...
    → M

In [30]:
avg_mape = np.mean(mape_scores)
print(f"\n📊 Rata-rata MAPE keseluruhan: {avg_mape:.4f}")


📊 Rata-rata MAPE keseluruhan: 0.1058


In [45]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Parameter sliding window
WINDOW_SIZE = 5

# Dictionary untuk menyimpan model dan prediksi
models = {}
predictions = []
mape_scores = []
predict_start = "2024-10-01"
predict_end = "2024-12-31"

# Buat daftar tanggal untuk prediksi
future_dates = pd.date_range(start=predict_start, end=predict_end)

# Fungsi untuk membuat dataset dengan sliding window
def create_windowed_data(df, target_col, window_size):
    X, y = [], []
    for i in range(len(df) - window_size):
        X.append(df.iloc[i:i+window_size].values.flatten())
        y.append(df.iloc[i + window_size][target_col])
    return np.array(X), np.array(y)

# Loop setiap komoditas dalam df_dict
for komoditas, df_komoditas in df_dict.items():
    print(f"\n🚀 Mengolah komoditas: {komoditas}")

    if "Date" in df_komoditas.columns:
        df_komoditas = df_komoditas.drop(columns=["Date"])

    for provinsi in df_komoditas.columns:
        print(f"  → Menggabungkan data untuk provinsi: {provinsi}...")
        
        df_prov = df_komoditas[[provinsi]].rename(columns={provinsi: 'y'})
        df_merged = pd.concat([df_prov, df_PCA_global, df_PCA_exchange], axis=1)
        
        # Buat data dengan windowing
        X, y = create_windowed_data(df_merged, 'y', WINDOW_SIZE)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
        
        # Inisialisasi dan latih model XGBoost
        model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
        model.fit(X_train, y_train)
        
        # Simpan model
        models[(komoditas, provinsi)] = model
        
        # Evaluasi model
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        mape = mean_absolute_percentage_error(y_test, y_pred)
        mape_scores.append(mape)
        print(f"    → MAE untuk {komoditas} - {provinsi}: {mae:.4f}")
        print(f"    → MAPE untuk {komoditas} - {provinsi}: {mape:.4f}")
        
        # Mulai prediksi ke masa depan dengan iterasi mirip LSTM
        last_known_data = X[-1].reshape(1, -1)
        future_preds = []

        for _ in range(len(future_dates)):
            next_pred = model.predict(last_known_data)[0]
            future_preds.append(next_pred)
            
            # Update sliding window dengan prediksi baru
            last_known_data = np.roll(last_known_data, -1)
            last_known_data[0, -1] = next_pred

        # Simpan hasil prediksi
        for date, pred in zip(future_dates, future_preds):
            predictions.append({
                'id': f"{komoditas}/{provinsi}/{date.strftime('%Y-%m-%d')}",
                'price': pred
            })

# Simpan hasil prediksi ke CSV
submission_df = pd.DataFrame(predictions)
submission_df.to_csv('submission_xgboost_lstm.csv', index=False)
print("\n✅ Hasil prediksi disimpan dalam 'submission_xgboost_lstm.csv'")


🚀 Mengolah komoditas: Bawang Merah
  → Menggabungkan data untuk provinsi: Aceh...
    → MAE untuk Bawang Merah - Aceh: 1545.1187
    → MAPE untuk Bawang Merah - Aceh: 0.0349
  → Menggabungkan data untuk provinsi: Bali...
    → MAE untuk Bawang Merah - Bali: 1015.7060
    → MAPE untuk Bawang Merah - Bali: 0.0309
  → Menggabungkan data untuk provinsi: Banten...
    → MAE untuk Bawang Merah - Banten: 1671.7198
    → MAPE untuk Bawang Merah - Banten: 0.0472
  → Menggabungkan data untuk provinsi: Bengkulu...
    → MAE untuk Bawang Merah - Bengkulu: 1294.0853
    → MAPE untuk Bawang Merah - Bengkulu: 0.0311
  → Menggabungkan data untuk provinsi: DI Yogyakarta...
    → MAE untuk Bawang Merah - DI Yogyakarta: 1669.4027
    → MAPE untuk Bawang Merah - DI Yogyakarta: 0.0496
  → Menggabungkan data untuk provinsi: DKI Jakarta...
    → MAE untuk Bawang Merah - DKI Jakarta: 3998.1191
    → MAPE untuk Bawang Merah - DKI Jakarta: 0.0844
  → Menggabungkan data untuk provinsi: Gorontalo...
    → MAE un

In [46]:
avg_mape = np.mean(mape_scores)
print(f"\n📊 Rata-rata MAPE keseluruhan: {avg_mape:.4f}")


📊 Rata-rata MAPE keseluruhan: 0.0269



🚀 Mengolah komoditas: Bawang Merah
  → Menggabungkan data untuk provinsi: Aceh...
    → MAE untuk Bawang Merah - Aceh: 3948.3149
    → MAPE untuk Bawang Merah - Aceh: 0.0903
  → Menggabungkan data untuk provinsi: Bali...
    → MAE untuk Bawang Merah - Bali: 1363.0883
    → MAPE untuk Bawang Merah - Bali: 0.0448
  → Menggabungkan data untuk provinsi: Banten...
    → MAE untuk Bawang Merah - Banten: 2113.3640
    → MAPE untuk Bawang Merah - Banten: 0.0560
  → Menggabungkan data untuk provinsi: Bengkulu...
    → MAE untuk Bawang Merah - Bengkulu: 2725.0368
    → MAPE untuk Bawang Merah - Bengkulu: 0.0795
  → Menggabungkan data untuk provinsi: DI Yogyakarta...
    → MAE untuk Bawang Merah - DI Yogyakarta: 2406.2238
    → MAPE untuk Bawang Merah - DI Yogyakarta: 0.0870
  → Menggabungkan data untuk provinsi: DKI Jakarta...
    → MAE untuk Bawang Merah - DKI Jakarta: 3893.3062
    → MAPE untuk Bawang Merah - DKI Jakarta: 0.0935
  → Menggabungkan data untuk provinsi: Gorontalo...
    → MAE un


🚀 Mengolah komoditas: Bawang Merah
  → Menggabungkan data untuk provinsi: Aceh...


/tmp/ipykernel_249124/3800958647.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_prov["MA7"] = df_prov["y"].rolling(window=7, min_periods=1).mean().shift(1).fillna(method='bfill')


    → Best MAPE untuk Bawang Merah - Aceh: 0.0267
  → Menggabungkan data untuk provinsi: Bali...


/tmp/ipykernel_249124/3800958647.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_prov["MA7"] = df_prov["y"].rolling(window=7, min_periods=1).mean().shift(1).fillna(method='bfill')


    → Best MAPE untuk Bawang Merah - Bali: 0.0364
  → Menggabungkan data untuk provinsi: Banten...


...

    → Best MAPE untuk Tepung Terigu (Curah) - Sumatera Barat: 0.0129
  → Menggabungkan data untuk provinsi: Sumatera Selatan...


/tmp/ipykernel_249124/3800958647.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_prov["MA7"] = df_prov["y"].rolling(window=7, min_periods=1).mean().shift(1).fillna(method='bfill')


    → Best MAPE untuk Tepung Terigu (Curah) - Sumatera Selatan: 0.0096
  → Menggabungkan data untuk provinsi: Sumatera Utara...


/tmp/ipykernel_249124/3800958647.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_prov["MA7"] = df_prov["y"].rolling(window=7, min_periods=1).mean().shift(1).fillna(method='bfill')


    → Best MAPE untuk Tepung Terigu (Curah) - Sumatera Utara: 0.0095

✅ Hasil prediksi disimpan dalam 'submission_xgboost_ma_rolling.csv'


In [42]:
avg_mape = np.mean(mape_scores)avg_mape = np.mean(mape_scores)
print(f"\n📊 Rata-rata MAPE keseluruhan: {avg_mape:.4f}")
print(f"\n📊 Rata-rata MAPE keseluruhan: {avg_mape:.4f}")


📊 Rata-rata MAPE keseluruhan: nan


/home/sirius/Documents/DS/myenv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/sirius/Documents/DS/myenv/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



🚀 Mengolah komoditas: Bawang Merah
  → Menggabungkan data untuk provinsi: Aceh...


ValueError: Cannot have number of folds=6 greater than the number of samples=0.

In [57]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler


SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dictionary untuk menyimpan model dan prediksi
models = {}
predictions = []
mape_scores = []
predict_start = "2024-10-01"
predict_end = "2024-12-31"

# Buat daftar tanggal untuk prediksi
future_dates = pd.date_range(start=predict_start, end=predict_end)

# Definisikan model LSTM
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

# Fungsi untuk melatih model LSTM
def train_lstm(model, train_loader, val_loader, criterion, optimizer, num_epochs=100):
    model.train()
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # Backward pass dan optimasi
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        val_mape = 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                val_loss += criterion(outputs, targets).item()
                
                # Hitung MAPE
                y_pred = scaler_y.inverse_transform(outputs.cpu().numpy())
                y_true = scaler_y.inverse_transform(targets.cpu().numpy())
                val_mape += mean_absolute_percentage_error(y_true, y_pred)
        
        # Hitung rata-rata loss dan MAPE
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        val_mape /= len(val_loader)
        
        # Tampilkan hasil setiap epoch
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val MAPE: {val_mape:.4f}')

# Loop setiap komoditas dalam df_dict
for komoditas, df_komoditas in df_dict.items():
    print(f"\n🚀 Mengolah komoditas: {komoditas}")

    # Drop kolom "Date" jika ada
    if "Date" in df_komoditas.columns:
        df_komoditas = df_komoditas.drop(columns=["Date"])

    # Loop setiap provinsi dalam df_komoditas
    for provinsi in df_komoditas.columns:
        print(f"  → Menggabungkan data untuk provinsi: {provinsi}...")

        # Ambil hanya data provinsi tersebut
        df_prov = df_komoditas[[provinsi]].rename(columns={provinsi: 'y'})

        # Gabungkan dengan PCA Global dan PCA Exchange
        df_merged = pd.concat([df_prov, df_PCA_global, df_PCA_exchange], axis=1)

        # Pisahkan fitur (X) dan target harga (y)
        X = df_merged.iloc[:, 1:].values  # Semua kolom kecuali harga komoditas
        y = df_merged.iloc[:, 0].values   # Kolom pertama sebagai target

        # Normalisasi data
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        X = scaler_X.fit_transform(X)
        y = scaler_y.fit_transform(y.reshape(-1, 1))

        # Pisahkan training & testing
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

        # Konversi ke PyTorch tensors
        X_train = torch.FloatTensor(X_train).unsqueeze(1)
        y_train = torch.FloatTensor(y_train)
        X_test = torch.FloatTensor(X_test).unsqueeze(1)
        y_test = torch.FloatTensor(y_test)

        # Buat DataLoader
        train_data = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_data, batch_size=32, shuffle=False)
        val_data = TensorDataset(X_test, y_test)
        val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

        # Inisialisasi model LSTM
        input_size = X_train.shape[2]
        hidden_size = 50
        num_layers = 2
        output_size = 1
        model = LSTMModel(input_size, hidden_size, num_layers, output_size).to(device)

        # Definisikan loss function dan optimizer
        criterion = nn.HuberLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.01)

        # Latih model
        train_lstm(model, train_loader, val_loader, criterion, optimizer, num_epochs=100)

        # Simpan model
        models[(komoditas, provinsi)] = model

        # Evaluasi model
        model.eval()
        with torch.no_grad():
            y_pred = model(X_test.to(device))
            y_pred = scaler_y.inverse_transform(y_pred.cpu().numpy())
            y_test_actual = scaler_y.inverse_transform(y_test.numpy())
            
            mae = mean_absolute_error(y_test_actual, y_pred)
            mape = mean_absolute_percentage_error(y_test_actual, y_pred)
            mape_scores.append(mape)  # Simpan MAPE
            print(f"    → MAE untuk {komoditas} - {provinsi}: {mae:.4f}")
            print(f"    → MAPE untuk {komoditas} - {provinsi}: {mape:.4f}")

        # Mulai prediksi ke masa depan dengan iterasi (mirip LSTM)
        last_known_data = X[-1, :].reshape(1, 1, -1)  # Ambil data terakhir
        last_known_data = torch.FloatTensor(last_known_data).to(device)
        future_preds = []

        for _ in range(len(future_dates)):
            with torch.no_grad():
                next_pred = model(last_known_data).cpu().numpy()[0][0]
                next_pred = scaler_y.inverse_transform(np.array([[next_pred]]))[0][0]
                future_preds.append(next_pred)

                # Update last_known_data → gantikan harga sebelumnya dengan prediksi baru
                last_known_data = torch.roll(last_known_data, -1, dims=2)
                
                # Konversi nilai prediksi ke tensor dan pastikan berada di perangkat yang sama
                next_pred_scaled = scaler_y.transform(np.array([[next_pred]]))[0][0]
                next_pred_tensor = torch.tensor([next_pred_scaled], dtype=torch.float32).to(device)
                
                last_known_data[0, 0, -1] = next_pred_tensor

        # Simpan hasil prediksi ke dalam list
        for date, pred in zip(future_dates, future_preds):
            predictions.append({
                'id': f"{komoditas}/{provinsi}/{date.strftime('%Y-%m-%d')}",
                'price': pred  
            })

# Hitung rata-rata MAPE
avg_mape = np.mean(mape_scores)
print(f"\n📊 Rata-rata MAPE untuk semua model: {avg_mape:.4f}")

# Simpan hasil prediksi ke CSV
submission_df = pd.DataFrame(predictions)
submission_df.to_csv('submission_lstm_pytorch.csv', index=False)
print("\n✅ Hasil prediksi disimpan dalam 'submission_lstm_pytorch.csv'")


🚀 Mengolah komoditas: Bawang Merah
  → Menggabungkan data untuk provinsi: Aceh...
Epoch [1/100], Train Loss: 0.2746, Val Loss: 0.8691, Val MAPE: 0.2617
Epoch [2/100], Train Loss: 0.2314, Val Loss: 0.8989, Val MAPE: 0.2686
Epoch [3/100], Train Loss: 0.1477, Val Loss: 0.8883, Val MAPE: 0.2866
Epoch [4/100], Train Loss: 0.1927, Val Loss: 0.8313, Val MAPE: 0.2695
Epoch [5/100], Train Loss: 0.2460, Val Loss: 0.8088, Val MAPE: 0.2382
Epoch [6/100], Train Loss: 0.1892, Val Loss: 0.8650, Val MAPE: 0.2519
Epoch [7/100], Train Loss: 0.1271, Val Loss: 0.9495, Val MAPE: 0.2963
Epoch [8/100], Train Loss: 0.1710, Val Loss: 0.8973, Val MAPE: 0.2757
Epoch [9/100], Train Loss: 0.1512, Val Loss: 0.9135, Val MAPE: 0.2842
Epoch [10/100], Train Loss: 0.1486, Val Loss: 0.8720, Val MAPE: 0.2825
Epoch [11/100], Train Loss: 0.1880, Val Loss: 0.8892, Val MAPE: 0.2744
Epoch [12/100], Train Loss: 0.0941, Val Loss: 1.0180, Val MAPE: 0.2935
Epoch [13/100], Train Loss: 0.1198, Val Loss: 0.9942, Val MAPE: 0.2998
Epo

KeyboardInterrupt: 